## 1. Installation & Setup

In [ ]:
# Install required packages
!pip install yt-dlp
!pip install spacy
!pip install nltk
!pip install pandas

# Download spaCy English model
!python -m spacy download en_core_web_sm

## 2. Import Libraries

In [ ]:
import os
import re
import json
import subprocess
from pathlib import Path
from typing import List, Dict, Tuple, Optional
import pandas as pd

# NLP libraries
import spacy
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize, sent_tokenize

# Download required NLTK data
nltk.download('punkt', quiet=True)
nltk.download('stopwords', quiet=True)
nltk.download('punkt_tab', quiet=True)

print("✓ All libraries imported successfully!")

## 3. Define the ElonMuskQAGenerator Class

In [ ]:
class ElonMuskQAGenerator:
    """Process Elon Musk interview transcripts into Q&A pairs."""
    
    def __init__(self, output_dir: str = "elon_musk_data"):
        """
        Initialize the Q&A generator.
        
        Args:
            output_dir: Directory to store downloaded files and processed data
        """
        self.output_dir = Path(output_dir)
        self.output_dir.mkdir(exist_ok=True)
        
        # Create subdirectories
        self.audio_dir = self.output_dir / "audio"
        self.transcript_dir = self.output_dir / "transcripts"
        self.processed_dir = self.output_dir / "processed"
        
        for directory in [self.audio_dir, self.transcript_dir, self.processed_dir]:
            directory.mkdir(exist_ok=True)
        
        # Load spaCy model
        self.nlp = spacy.load("en_core_web_sm")
        
        # Get English stopwords
        self.stop_words = set(stopwords.words('english'))
        
        # Common interviewer phrases to identify questions
        self.question_indicators = [
            r'\bwhat\b', r'\bhow\b', r'\bwhy\b', r'\bwhen\b', r'\bwhere\b',
            r'\bwho\b', r'\bwhich\b', r'\bcan you\b', r'\bcould you\b',
            r'\bwould you\b', r'\bdo you\b', r'\bdid you\b', r'\bare you\b',
            r'\bwill you\b', r'\btell me\b', r'\btalk about\b'
        ]
        
    def download_video(self, url: str, video_id: Optional[str] = None) -> Tuple[str, str]:
        """Download audio and subtitles from YouTube video."""
        if video_id is None:
            video_id = self._extract_video_id(url)
        
        audio_path = self.audio_dir / f"{video_id}.mp3"
        subtitle_path = self.transcript_dir / f"{video_id}.vtt"
        
        print(f"\nDownloading video: {video_id}")
        print(f"URL: {url}")
        
        # Download audio
        if not audio_path.exists():
            print("Downloading audio...")
            audio_cmd = [
                "yt-dlp",
                "-x",
                "--audio-format", "mp3",
                "--output", str(audio_path),
                url
            ]
            subprocess.run(audio_cmd, check=True)
            print(f"✓ Audio saved to: {audio_path}")
        else:
            print(f"✓ Audio already exists: {audio_path}")
        
        # Download subtitles
        if not subtitle_path.exists():
            print("Downloading subtitles...")
            subtitle_cmd = [
                "yt-dlp",
                "--write-auto-subs",
                "--sub-lang", "en",
                "--sub-format", "vtt",
                "--skip-download",
                "--output", str(self.transcript_dir / video_id),
                url
            ]
            subprocess.run(subtitle_cmd, check=True)
            
            # Find and rename VTT file
            vtt_files = list(self.transcript_dir.glob(f"{video_id}*.vtt"))
            if vtt_files:
                vtt_files[0].rename(subtitle_path)
                print(f"✓ Subtitles saved to: {subtitle_path}")
            else:
                print("⚠ Warning: Could not download subtitles")
                subtitle_path = None
        else:
            print(f"✓ Subtitles already exist: {subtitle_path}")
        
        return str(audio_path), str(subtitle_path) if subtitle_path else None
    
    def _extract_video_id(self, url: str) -> str:
        """Extract video ID from YouTube URL."""
        patterns = [
            r'(?:youtube\.com\/watch\?v=|youtu\.be\/)([^&\n?]+)',
            r'youtube\.com\/embed\/([^&\n?]+)',
        ]
        
        for pattern in patterns:
            match = re.search(pattern, url)
            if match:
                return match.group(1)
        
        return str(hash(url))[:10]
    
    def parse_vtt(self, vtt_path: str) -> str:
        """Parse VTT subtitle file and extract clean text."""
        with open(vtt_path, 'r', encoding='utf-8') as f:
            content = f.read()
        
        # Remove VTT header
        content = re.sub(r'^WEBVTT.*?\n\n', '', content, flags=re.DOTALL)
        
        # Extract text, skip timestamps
        lines = []
        for line in content.split('\n'):
            if '-->' in line or re.match(r'^\d+$', line.strip()):
                continue
            if line.strip() and not line.startswith('<'):
                clean_line = re.sub(r'<[^>]+>', '', line)
                lines.append(clean_line.strip())
        
        text = ' '.join(lines)
        text = self._remove_duplicate_words(text)
        
        return text
    
    def _remove_duplicate_words(self, text: str) -> str:
        """Remove consecutive duplicate words."""
        words = text.split()
        cleaned = []
        prev_word = None
        
        for word in words:
            if word.lower() != prev_word:
                cleaned.append(word)
            prev_word = word.lower()
        
        return ' '.join(cleaned)
    
    def identify_speakers(self, text: str) -> List[Dict[str, str]]:
        """Identify speaker segments using heuristics."""
        sentences = sent_tokenize(text)
        segments = []
        
        for sent in sentences:
            sent_lower = sent.lower()
            
            # Check if sentence is a question
            is_question = (
                sent.strip().endswith('?') or
                any(re.search(pattern, sent_lower) for pattern in self.question_indicators)
            )
            
            speaker = "interviewer" if is_question else "elon"
            
            segments.append({
                "speaker": speaker,
                "text": sent.strip()
            })
        
        return segments
    
    def extract_qa_pairs(self, segments: List[Dict[str, str]]) -> List[Dict[str, str]]:
        """Extract question-answer pairs from segments."""
        qa_pairs = []
        current_question = None
        current_answer = []
        
        for segment in segments:
            if segment['speaker'] == 'interviewer':
                if current_question and current_answer:
                    qa_pairs.append({
                        "question": current_question,
                        "answer": ' '.join(current_answer)
                    })
                
                current_question = segment['text']
                current_answer = []
            
            elif segment['speaker'] == 'elon':
                if current_question:
                    current_answer.append(segment['text'])
        
        if current_question and current_answer:
            qa_pairs.append({
                "question": current_question,
                "answer": ' '.join(current_answer)
            })
        
        return qa_pairs
    
    def preprocess_text(self, text: str, remove_stopwords: bool = False) -> str:
        """Preprocess text using spaCy (lemmatization, tokenization)."""
        doc = self.nlp(text)
        
        tokens = []
        for token in doc:
            if token.is_punct or token.is_space:
                continue
            
            if remove_stopwords and token.text.lower() in self.stop_words:
                continue
            
            tokens.append(token.lemma_)
        
        return ' '.join(tokens)
    
    def process_video(self, url: str, video_id: Optional[str] = None) -> List[Dict[str, str]]:
        """Process a single video end-to-end."""
        audio_path, subtitle_path = self.download_video(url, video_id)
        
        if not subtitle_path:
            print(f"⚠ Skipping video due to missing subtitles")
            return []
        
        print("\nParsing transcript...")
        transcript = self.parse_vtt(subtitle_path)
        
        print("Identifying speakers...")
        segments = self.identify_speakers(transcript)
        
        print("Extracting Q&A pairs...")
        qa_pairs = self.extract_qa_pairs(segments)
        
        print(f"✓ Extracted {len(qa_pairs)} Q&A pairs")
        
        return qa_pairs
    
    def process_multiple_videos(self, urls: List[str]) -> pd.DataFrame:
        """Process multiple videos and combine into dataset."""
        all_qa_pairs = []
        
        for i, url in enumerate(urls, 1):
            print(f"\n{'='*80}")
            print(f"Processing video {i}/{len(urls)}")
            print(f"{'='*80}")
            
            try:
                qa_pairs = self.process_video(url)
                all_qa_pairs.extend(qa_pairs)
            except Exception as e:
                print(f"✗ Error processing video: {e}")
                continue
        
        df = pd.DataFrame(all_qa_pairs)
        
        print(f"\nRemoving duplicate Q&A pairs...")
        original_count = len(df)
        df = df.drop_duplicates(subset=['question', 'answer'])
        print(f"Removed {original_count - len(df)} duplicates")
        
        df = df[df['answer'].str.split().str.len() > 3]
        
        print(f"\n✓ Final dataset contains {len(df)} Q&A pairs")
        
        return df
    
    def save_dataset(self, df: pd.DataFrame, filename: str = "elon_musk_qa_dataset"):
        """Save dataset in multiple formats."""
        csv_path = self.processed_dir / f"{filename}.csv"
        df.to_csv(csv_path, index=False, encoding='utf-8')
        print(f"✓ Saved CSV: {csv_path}")
        
        json_path = self.processed_dir / f"{filename}.json"
        df.to_json(json_path, orient='records', indent=2, force_ascii=False)
        print(f"✓ Saved JSON: {json_path}")
        
        jsonl_path = self.processed_dir / f"{filename}.jsonl"
        df.to_json(jsonl_path, orient='records', lines=True, force_ascii=False)
        print(f"✓ Saved JSONL: {jsonl_path}")
        
        return csv_path, json_path, jsonl_path

print("✓ ElonMuskQAGenerator class defined successfully!")

## 4. Configure YouTube URLs

In [ ]:
youtube_urls = [
    "https://www.youtube.com/watch?v=JGdbFGANapk",
    "https://www.youtube.com/watch?v=YqDehngsBHw",
    "https://www.youtube.com/watch?v=S_vpv4I27hs",
    "https://www.youtube.com/watch?v=QbNODZwQQuw"
]

print(f"Configured {len(youtube_urls)} video(s) for processing")

## 5. Initialize the Generator

In [ ]:
# Initialize the Q&A generator
generator = ElonMuskQAGenerator(output_dir="elon_musk_data")

print("✓ Generator initialized!")
print(f"Output directory: {generator.output_dir.absolute()}")
print(f"Audio directory: {generator.audio_dir.absolute()}")
print(f"Transcript directory: {generator.transcript_dir.absolute()}")
print(f"Processed data directory: {generator.processed_dir.absolute()}")

## 6. Process Videos

1. Download audio and subtitles for each video
2. Parse the VTT transcripts
3. Identify questions vs. answers
4. Create Q&A pairs
5. Remove duplicates

In [ ]:
# Process all videos
if not youtube_urls:
    print("⚠ Please add YouTube URLs to the 'youtube_urls' list above!")
else:
    df = generator.process_multiple_videos(youtube_urls)
    print("\n✓ Processing complete!")

## 7. Explore the Dataset

In [ ]:
# Display dataset information
if 'df' in locals() and not df.empty:
    print(f"Total Q&A pairs: {len(df)}")
    print(f"\nDataset shape: {df.shape}")
    print(f"\nColumn names: {df.columns.tolist()}")
    print(f"\nFirst few rows:")
    display(df.head())
else:
    print("No data available. Please process videos first.")

In [ ]:
# Display statistics
if 'df' in locals() and not df.empty:
    print("Dataset Statistics:")
    print("=" * 50)
    print(f"\nAverage question length: {df['question'].str.split().str.len().mean():.1f} words")
    print(f"Average answer length: {df['answer'].str.split().str.len().mean():.1f} words")
    print(f"Median answer length: {df['answer'].str.split().str.len().median():.1f} words")
    print(f"Longest answer: {df['answer'].str.split().str.len().max()} words")
    print(f"Shortest answer: {df['answer'].str.split().str.len().min()} words")

In [ ]:
# Display sample Q&A pairs
if 'df' in locals() and not df.empty:
    print("Sample Q&A Pairs:")
    print("=" * 80)
    
    for i, row in df.sample(min(5, len(df))).iterrows():
        print(f"\nQ: {row['question']}")
        answer = row['answer'][:300] + "..." if len(row['answer']) > 300 else row['answer']
        print(f"A: {answer}")
        print("-" * 80)

## 8. Save the Dataset

In [ ]:
# Save dataset in multiple formats
if 'df' in locals() and not df.empty:
    csv_path, json_path, jsonl_path = generator.save_dataset(df, filename="elon_musk_qa_dataset")
    
    print("\n✓ Dataset saved successfully!")
    print(f"\nYou can find your files at:")
    print(f"  - CSV: {csv_path}")
    print(f"  - JSON: {json_path}")
    print(f"  - JSONL: {jsonl_path}")
else:
    print("No data to save. Please process videos first.")

## 9. Apply NLP Processing

Apply stopword removal and lemmatization to individual texts.

In [ ]:
if 'df' in locals() and not df.empty:
    sample_answer = df.iloc[0]['answer']
    
    print("Original answer:")
    print(sample_answer[:300])
    print("\n" + "="*80 + "\n")
    
    # Without stopword removal (recommended for conversational data)
    processed_without_stopwords = generator.preprocess_text(sample_answer, remove_stopwords=False)
    print("Processed (lemmatized, no stopword removal):")
    print(processed_without_stopwords[:300])
    print("\n" + "="*80 + "\n")
    
    # With stopword removal (may lose conversational tone)
    processed_with_stopwords = generator.preprocess_text(sample_answer, remove_stopwords=True)
    print("Processed (lemmatized, with stopword removal):")
    print(processed_with_stopwords[:300])

## 10. Export for Training

Prepare the dataset in a format suitable for training conversational models.

In [ ]:
# Create a training-ready format
if 'df' in locals() and not df.empty:
    training_data = []
    for _, row in df.iterrows():
        training_data.append({
            "messages": [
                {"role": "user", "content": row['question']},
                {"role": "assistant", "content": row['answer']}
            ]
        })
    
    # Save training format
    training_path = generator.processed_dir / "training_data.jsonl"
    with open(training_path, 'w', encoding='utf-8') as f:
        for item in training_data:
            f.write(json.dumps(item, ensure_ascii=False) + '\n')
    
    print(f"✓ Training data saved to: {training_path}")
    print(f"Total training examples: {len(training_data)}")